In [24]:
# Import Python's built-in tools for file paths and CSV reading
from pathlib import Path
import csv

# Point to the CSV file inside the project's data folder
DATA_PATH = Path("data") / "electronics_sales.csv"

# Open the CSV and read each sale as a dictionary
with DATA_PATH.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
    column_names = reader.fieldnames

# Show how many sales were read and which columns are present
print(f"Rows read: {len(rows)}")
print(f"Columns: {column_names}")

Rows read: 50
Columns: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']


In [25]:
# Check each row for quantity or price values that aren't plain digits
for row in rows:
    quantity_text = row["quantity"].strip()
    price_text = row["unit_price_kes"].strip()

    # Show the sale if either value needs a closer look
    if not quantity_text.isdigit() or not price_text.isdigit():
        print(
            row["sale_id"],
            f"quantity={quantity_text!r}",
            f"unit_price_kes={price_text!r}"
        )

S-3006 quantity='two' unit_price_kes='3199'
S-3017 quantity='4' unit_price_kes='38,999'
S-3029 quantity='1' unit_price_kes='KES 18999'
S-3030 quantity='1' unit_price_kes='N/A'
S-3035 quantity='' unit_price_kes='13499'
S-3039 quantity='1' unit_price_kes=''


In [26]:
def parse_number(value: str) -> float | None:
    # Remove extra spaces around the raw value
    text = value.strip()

    # Treat blank strings and common missing-value labels as missing
    if text == "" or text.upper() in {"N/A", "NA", "NONE", "NULL"}:
        return None

    # Repair this unambiguous quantity written as a word
    if text.lower() == "two":
        return 2.0

    # Remove a currency label at the start of the value
    if text.upper().startswith("KES"):
        text = text[3:].strip()

    # Remove commas only when they form valid thousands groups
    if "," in text:
        groups = text.split(",")
        valid_thousands = (
            groups[0].isdigit()
            and 1 <= len(groups[0]) <= 3
            and all(group.isdigit() and len(group) == 3 for group in groups[1:])
        )

        # Mark incorrectly grouped values as untrusted
        if not valid_thousands:
            return None

        text = "".join(groups)

    # Convert the remaining numeric text to a number
    try:
        return float(text)
    except ValueError:
        return None


# Check the parser on normal, repairable, and missing values
test_values = ["3199", "38,999", "KES 18999", "two", "N/A", ""]

for value in test_values:
    print(f"{value!r} -> {parse_number(value)}")

'3199' -> 3199.0
'38,999' -> 38999.0
'KES 18999' -> 18999.0
'two' -> 2.0
'N/A' -> None
'' -> None


In [27]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    # Store usable sales and sales that need to be skipped
    clean_rows = []
    skipped_rows = []

    # Convert each sale's quantity and unit price
    for row in rows:
        quantity_value = parse_number(row["quantity"])
        price_value = parse_number(row["unit_price_kes"])
        reasons = []

        # Record quantity problems
        if quantity_value is None:
            reasons.append(
                f"quantity is blank or unrecognized: {row['quantity']!r}"
            )
        elif quantity_value <= 0 or not quantity_value.is_integer():
            reasons.append("quantity must be a positive whole number")

        # Record price problems
        if price_value is None:
            reasons.append(
                f"unit price is blank or unrecognized: {row['unit_price_kes']!r}"
            )
        elif price_value <= 0:
            reasons.append("unit price must be positive")

        # Keep the original row and its reason if any required value is unusable
        if reasons:
            skipped_row = row.copy()
            skipped_row["reason"] = "; ".join(reasons)
            skipped_rows.append(skipped_row)
            continue

        # Copy valid rows, replace their numeric fields, and calculate revenue
        clean_row = row.copy()
        clean_row["quantity"] = int(quantity_value)
        clean_row["unit_price_kes"] = price_value
        clean_row["revenue"] = int(quantity_value) * price_value
        clean_rows.append(clean_row)

    # Return the usable sales and the skipped sales separately
    return clean_rows, skipped_rows


# Convert the raw sales and summarize the skipped rows
clean_rows, skipped_rows = convert_rows(rows)

print(f"Rows used: {len(clean_rows)}")
print(f"Rows skipped: {len(skipped_rows)}")

for skipped_row in skipped_rows:
    print(f"{skipped_row['sale_id']}: {skipped_row['reason']}")

Rows used: 47
Rows skipped: 3
S-3030: unit price is blank or unrecognized: 'N/A'
S-3035: quantity is blank or unrecognized: ''
S-3039: unit price is blank or unrecognized: ''


In [28]:
def total_revenue(rows: list[dict]) -> float:
    # Add the revenue from every usable sale
    return sum(row["revenue"] for row in rows)


# Calculate and format total revenue in KES
revenue_total = total_revenue(clean_rows)
print(f"Total revenue (KES): {revenue_total:,.2f}")

Total revenue (KES): 1,317,210.00


In [29]:
def revenue_by(rows: list[dict], field: str) -> dict[str, float]:
    # Create a plain dictionary for the totals
    totals = {}

    # Add each sale's revenue to the matching product, store, or payment method
    for row in rows:
        group_name = row[field]
        totals[group_name] = totals.get(group_name, 0.0) + row["revenue"]

    return totals


# Calculate revenue totals for each requested category
product_revenue = revenue_by(clean_rows, "product")
store_revenue = revenue_by(clean_rows, "store")
payment_revenue = revenue_by(clean_rows, "payment_method")

# Display each breakdown from highest revenue to lowest
for category_name, totals in [
    ("Product", product_revenue),
    ("Store", store_revenue),
    ("Payment method", payment_revenue),
]:
    print(f"\nRevenue by {category_name.lower()} (KES)")

    # Sort by revenue descending, then name alphabetically to break ties
    for name, amount in sorted(
        totals.items(),
        key=lambda item: (-item[1], item[0])
    ):
        print(f"  {name}: {amount:,.2f}")


Revenue by product (KES)
  HP 15 Laptop: 389,994.00
  Hisense 43in Smart TV: 272,993.00
  Lenovo Tab M10: 199,992.00
  Tecno Spark 20: 161,988.00
  Infinix Hot 40: 63,996.00
  Samsung Galaxy A15: 56,997.00
  JBL Go 3 Speaker: 49,491.00
  Anker Power Bank 20000: 38,691.00
  Xiaomi Redmi Buds: 31,990.00
  Oraimo FreePods: 19,593.00
  USB-C Charger 25W: 19,487.00
  Safaricom Neon Ray: 11,998.00

Revenue by store (KES)
  Kisumu Mega Plaza: 449,678.00
  Nairobi CBD: 337,483.00
  Westgate: 235,979.00
  Mombasa City Mall: 154,082.00
  Nakuru Westside: 139,988.00

Revenue by payment method (KES)
  M-Pesa: 738,568.00
  Cash: 310,773.00
  Card: 267,869.00


In [30]:
def top_n(totals: dict[str, float], n: int = 3) -> list[tuple[str, float]]:
    # Sort by revenue descending, then name alphabetically for ties
    ranked = sorted(
        totals.items(),
        key=lambda item: (-item[1], item[0])
    )

    # Return only the first n ranked products
    return ranked[:n]


# Get and display the three highest-revenue products
top_products = top_n(product_revenue, 3)

print("Top 3 products")
for rank, (product, amount) in enumerate(top_products, start=1):
    print(f"  {rank}. {product} - KES {amount:,.2f}")

Top 3 products
  1. HP 15 Laptop - KES 389,994.00
  2. Hisense 43in Smart TV - KES 272,993.00
  3. Lenovo Tab M10 - KES 199,992.00


In [31]:
def high_value(rows: list[dict], threshold: float) -> list[dict]:
    # Keep only sales at or above the threshold
    return [row for row in rows if row["revenue"] >= threshold]


# Set the follow-up threshold and find qualifying sales
threshold = 50_000
high_value_sales = high_value(clean_rows, threshold)

# Count qualifying sales at each store with a dictionary comprehension
high_value_count_by_store = {
    store: len([
        row for row in high_value_sales
        if row["store"] == store
    ])
    for store in sorted({row["store"] for row in clean_rows})
}

# Display the qualifying transactions
print(f"High-value transactions (>= KES {threshold:,.0f}): {len(high_value_sales)}")

for sale in high_value_sales:
    print(
        f"  {sale['sale_id']}  {sale['sale_date']}  {sale['store']}  "
        f"{sale['product']}  KES {sale['revenue']:,.2f}"
    )

# Display the qualifying-sale count for every store
print("\nHigh-value sales by store")
for store, count in high_value_count_by_store.items():
    print(f"  {store}: {count}")

High-value transactions (>= KES 50,000): 5
  S-3013  2026-09-08  Westgate  Lenovo Tab M10  KES 99,996.00
  S-3017  2026-09-10  Kisumu Mega Plaza  Hisense 43in Smart TV  KES 155,996.00
  S-3022  2026-09-13  Nairobi CBD  HP 15 Laptop  KES 259,996.00
  S-3036  2026-09-21  Nakuru Westside  Tecno Spark 20  KES 53,996.00
  S-3048  2026-09-30  Kisumu Mega Plaza  HP 15 Laptop  KES 129,998.00

High-value sales by store
  Kisumu Mega Plaza: 2
  Mombasa City Mall: 0
  Nairobi CBD: 1
  Nakuru Westside: 1
  Westgate: 1


In [32]:
def mean(values: list[float]) -> float:
    # Require at least one value to calculate an average
    if not values:
        raise ValueError("mean() needs at least one value")

    # Divide the sum by the number of values
    return sum(values) / len(values)


def median(values: list[float]) -> float:
    # Require at least one value to find a middle value
    if not values:
        raise ValueError("median() needs at least one value")

    # Sort a copy so the original list remains unchanged
    ordered_values = sorted(values)
    middle_index = len(ordered_values) // 2

    # Return the middle value for an odd-sized list
    if len(ordered_values) % 2 == 1:
        return ordered_values[middle_index]

    # Average the two middle values for an even-sized list
    return (
        ordered_values[middle_index - 1] + ordered_values[middle_index]
    ) / 2


def minimum_maximum(values: list[float]) -> tuple[float, float]:
    # Require at least one value to find a minimum and maximum
    if not values:
        raise ValueError("minimum_maximum() needs at least one value")

    # Start both results at the first value
    smallest = values[0]
    largest = values[0]

    # Compare each remaining value with the current smallest and largest
    for value in values[1:]:
        if value < smallest:
            smallest = value
        if value > largest:
            largest = value

    return smallest, largest


# Collect revenue from each usable sale and calculate its statistics
transaction_revenues = [row["revenue"] for row in clean_rows]
average_revenue = mean(transaction_revenues)
median_revenue = median(transaction_revenues)
minimum_revenue, maximum_revenue = minimum_maximum(transaction_revenues)

# Print all four statistics in KES
print("Transaction revenue statistics (KES)")
print(f"  mean   : {average_revenue:,.2f}")
print(f"  median : {median_revenue:,.2f}")
print(f"  min    : {minimum_revenue:,.2f}")
print(f"  max    : {maximum_revenue:,.2f}")

Transaction revenue statistics (KES)
  mean   : 28,025.74
  median : 10,998.00
  min    : 1,499.00
  max    : 259,996.00


In [34]:
%pip install pandas

  Using cached pandas-3.1.0-cp315-cp315-win_amd64.whl.metadata (18 kB)
  Using cached numpy-2.5.4-cp315-cp315-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.1.0-cp315-cp315-win_amd64.whl (10.9 MB)
Using cached numpy-2.5.4-cp315-cp315-win_amd64.whl (12.7 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
# Use pandas to independently verify the results from the standard-library analysis.
import math
import pandas as pd


def verify_with_pandas(path, my_results):
    # Read every CSV value as text so the existing parser handles missing markers consistently.
    pandas_rows = pd.read_csv(path, dtype=str, keep_default_na=False)

    # Apply the same number-cleaning function used by the main analysis.
    quantities = pd.to_numeric(
        pandas_rows["quantity"].map(parse_number),
        errors="coerce"
    )
    prices = pd.to_numeric(
        pandas_rows["unit_price_kes"].map(parse_number),
        errors="coerce"
    )

    # Keep only rows with positive prices and positive whole-number quantities.
    valid_rows = (
        quantities.notna()
        & prices.notna()
        & (quantities > 0)
        & ((quantities % 1) == 0)
        & (prices > 0)
    )

    # Calculate transaction revenue for the rows that passed the cleaning rules.
    cleaned = pandas_rows.loc[valid_rows].copy()
    cleaned["quantity_numeric"] = quantities.loc[valid_rows]
    cleaned["price_numeric"] = prices.loc[valid_rows]
    cleaned["revenue"] = cleaned["quantity_numeric"] * cleaned["price_numeric"]

    # Recalculate the headline numbers with pandas.
    pandas_total = cleaned["revenue"].sum()
    pandas_products = cleaned.groupby("product")["revenue"].sum().to_dict()
    pandas_top_three = sorted(
        pandas_products.items(),
        key=lambda item: (-item[1], item[0])
    )[:3]

    # Compare monetary values to within half a cent.
    def same_number(first, second):
        return math.isclose(
            float(first),
            float(second),
            rel_tol=0.0,
            abs_tol=0.005
        )

    # Compare the product totals, including product names.
    products_match = (
        my_results["product_revenue"].keys() == pandas_products.keys()
        and all(
            same_number(my_results["product_revenue"][product], pandas_products[product])
            for product in my_results["product_revenue"]
        )
    )

    # Compare product names and revenue for the top three.
    top_three_match = (
        len(my_results["top_products"]) == len(pandas_top_three)
        and all(
            expected_name == pandas_name
            and same_number(expected_revenue, pandas_revenue)
            for (expected_name, expected_revenue), (pandas_name, pandas_revenue)
            in zip(my_results["top_products"], pandas_top_three)
        )
    )

    # Collect each comparison so the report cell can use these results later.
    checks = {
        "Total revenue": same_number(my_results["total_revenue"], pandas_total),
        "Revenue by product": products_match,
        "Top three products": top_three_match,
        "Mean transaction revenue": same_number(
            my_results["mean_revenue"],
            cleaned["revenue"].mean()
        ),
        "Median transaction revenue": same_number(
            my_results["median_revenue"],
            cleaned["revenue"].median()
        ),
        "Minimum and maximum revenue": (
            same_number(my_results["minimum_revenue"], cleaned["revenue"].min())
            and same_number(my_results["maximum_revenue"], cleaned["revenue"].max())
        ),
    }

    # Print MATCH when the two methods agree, otherwise print MISMATCH.
    for result_name, matches in checks.items():
        status = "MATCH" if matches else "MISMATCH"
        print(f"{result_name}: {status}")

    return checks


# Gather the results from the standard-library analysis for comparison.
my_results = {
    "total_revenue": revenue_total,
    "product_revenue": product_revenue,
    "top_products": top_products,
    "mean_revenue": average_revenue,
    "median_revenue": median_revenue,
    "minimum_revenue": minimum_revenue,
    "maximum_revenue": maximum_revenue,
}

# Run the pandas verification against the project CSV.
verification_results = verify_with_pandas(DATA_PATH, my_results)

Total revenue: MATCH
Revenue by product: MATCH
Top three products: MATCH
Mean transaction revenue: MATCH
Median transaction revenue: MATCH
Minimum and maximum revenue: MATCH


In [36]:
# Build the final report, save it in the output folder, and display it.

from pathlib import Path

report_lines = [
    "TECH HUB KENYA — SEPTEMBER SALES REPORT",
    "=" * 42,
    f"Rows read: {len(rows)}",
    f"Rows included: {len(clean_rows)}",
    f"Rows skipped: {len(skipped_rows)}",
    "",
    f"Total revenue: KES {revenue_total:,.2f}",
    "",
    "TOP THREE PRODUCTS",
]

# List the three highest-revenue products.
for rank, (product, amount) in enumerate(top_products, start=1):
    report_lines.append(f"{rank}. {product}: KES {amount:,.2f}")

# Add revenue totals for each requested category.
for section_title, totals in [
    ("REVENUE BY PRODUCT", product_revenue),
    ("REVENUE BY STORE", store_revenue),
    ("REVENUE BY PAYMENT METHOD", payment_revenue),
]:
    report_lines.extend(["", section_title])

    for name, amount in sorted(
        totals.items(),
        key=lambda item: (-item[1], item[0])
    ):
        report_lines.append(f"{name}: KES {amount:,.2f}")

# List sales that meet the KES 50,000 follow-up threshold.
report_lines.extend([
    "",
    f"HIGH-VALUE SALES (at least KES {threshold:,.0f})",
])

for sale in high_value_sales:
    report_lines.append(
        f"{sale['sale_id']} | {sale['sale_date']} | {sale['store']} | "
        f"{sale['product']} | KES {sale['revenue']:,.2f}"
    )

# Show the high-value sale count for every store.
report_lines.extend(["", "HIGH-VALUE SALES BY STORE"])

for store, count in high_value_count_by_store.items():
    report_lines.append(f"{store}: {count}")

# Include transaction-level revenue statistics.
report_lines.extend([
    "",
    "TRANSACTION REVENUE STATISTICS",
    f"Mean: KES {average_revenue:,.2f}",
    f"Median: KES {median_revenue:,.2f}",
    f"Minimum: KES {minimum_revenue:,.2f}",
    f"Maximum: KES {maximum_revenue:,.2f}",
])

# Record each row excluded during cleaning and its reason.
report_lines.extend(["", "SKIPPED ROWS"])

for sale in skipped_rows:
    report_lines.append(f"{sale['sale_id']}: {sale['reason']}")

# Add the results of the pandas verification.
report_lines.extend(["", "PANDAS VERIFICATION"])

for result_name, matches in verification_results.items():
    status = "MATCH" if matches else "MISMATCH"
    report_lines.append(f"{result_name}: {status}")

# Save the report, creating the output folder if it is missing.
report_text = "\n".join(report_lines) + "\n"
report_path = Path("output") / "sales_report.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report_text, encoding="utf-8")

# Display the report and confirm where it was saved.
print(report_text)
print(f"Saved report to: {report_path}")

TECH HUB KENYA — SEPTEMBER SALES REPORT
Rows read: 50
Rows included: 47
Rows skipped: 3

Total revenue: KES 1,317,210.00

TOP THREE PRODUCTS
1. HP 15 Laptop: KES 389,994.00
2. Hisense 43in Smart TV: KES 272,993.00
3. Lenovo Tab M10: KES 199,992.00

REVENUE BY PRODUCT
HP 15 Laptop: KES 389,994.00
Hisense 43in Smart TV: KES 272,993.00
Lenovo Tab M10: KES 199,992.00
Tecno Spark 20: KES 161,988.00
Infinix Hot 40: KES 63,996.00
Samsung Galaxy A15: KES 56,997.00
JBL Go 3 Speaker: KES 49,491.00
Anker Power Bank 20000: KES 38,691.00
Xiaomi Redmi Buds: KES 31,990.00
Oraimo FreePods: KES 19,593.00
USB-C Charger 25W: KES 19,487.00
Safaricom Neon Ray: KES 11,998.00

REVENUE BY STORE
Kisumu Mega Plaza: KES 449,678.00
Nairobi CBD: KES 337,483.00
Westgate: KES 235,979.00
Mombasa City Mall: KES 154,082.00
Nakuru Westside: KES 139,988.00

REVENUE BY PAYMENT METHOD
M-Pesa: KES 738,568.00
Cash: KES 310,773.00
Card: KES 267,869.00

HIGH-VALUE SALES (at least KES 50,000)
S-3013 | 2026-09-08 | Westgate | Le